# **Project:** Develop a predictive model to accurately forecast hourly traffic volumes at different road junctions based on historical traffic data
## **Component 3**: Exploratory Data Analysis & Data Preparation
## **Things wanted to cover in component 3**: data loading, cleaning & pre-processing, aggregation, normalization/standardization, feature engineering (time-based, lag, and event features), and feature importance evaluation.

## 1. Import Required Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statistics as st
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

## 2. Load Raw Data

In [2]:
# Load the integrated traffic + weather + event dataset
df = pd.read_csv("/content/Integrated_Traffic_Weather_Event_Dataset_Yashas.csv",parse_dates=["DateTime"])
df.head()

,DateTime,Junction,Vehicles,ID,temperature_c,humidity_pct,precipitation_mm,rain_mm,wind_speed_kmh,is_holiday,is_event_day,is_special_day,hour,day_of_week,is_weekend,month,is_rush_hour
0,2015-11-01 00:00:00,1,15,20151101001,20.2,94.0,0.0,0.0,6.8,1,0,1,0,6,1,11,0
1,2015-11-01 00:00:00,2,6,20151101002,20.2,94.0,0.0,0.0,6.8,1,0,1,0,6,1,11,0
2,2015-11-01 00:00:00,3,9,20151101003,20.2,94.0,0.0,0.0,6.8,1,0,1,0,6,1,11,0
3,2015-11-01 01:00:00,3,7,20151101013,20.2,94.0,0.0,0.0,6.8,1,0,1,1,6,1,11,0
4,2015-11-01 01:00:00,1,13,20151101011,20.2,94.0,0.0,0.0,6.8,1,0,1,1,6,1,11,0


In [3]:
print(df.shape)
print(df.columns)

(48120, 17)
Index(['DateTime', 'Junction', 'Vehicles', 'ID', 'temperature_c',
       'humidity_pct', 'precipitation_mm', 'rain_mm', 'wind_speed_kmh',
       'is_holiday', 'is_event_day', 'is_special_day', 'hour', 'day_of_week',
       'is_weekend', 'month', 'is_rush_hour'],
      dtype='object')


In [4]:
df.dtypes

,0
DateTime,datetime64[ns]
Junction,int64
Vehicles,int64
ID,int64
temperature_c,float64
humidity_pct,float64
precipitation_mm,float64
rain_mm,float64
wind_speed_kmh,float64
is_holiday,int64


In [5]:
df.describe()

,DateTime,Junction,Vehicles,ID,temperature_c,humidity_pct,precipitation_mm,rain_mm,wind_speed_kmh,is_holiday,is_event_day,is_special_day,hour,day_of_week,is_weekend,month,is_rush_hour
count,48120,48120.000000,48120.000000,4.812000e+04,48120.000000,48120.000000,48120.000000,48120.000000,48120.000000,48120.000000,48120.000000,48120.000000,48120.000000,48120.000000,48120.000000,48120.000000,48120.000000
mean,2016-09-19 06:03:56.109725696,2.180549,22.791334,2.016330e+10,23.520648,64.901517,0.096687,0.096687,11.272421,0.055362,0.029426,0.082793,11.500000,2.996010,0.284289,5.884289,0.291667
min,2015-11-01 00:00:00,1.000000,1.000000,2.015110e+10,11.500000,9.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000
25%,2016-04-16 01:45:00,1.000000,9.000000,2.016042e+10,20.300000,46.000000,0.000000,0.000000,7.900000,0.000000,0.000000,0.000000,5.750000,1.000000,0.000000,3.000000,0.000000
50%,2016-09-30 03:30:00,2.000000,15.000000,2.016093e+10,23.100000,68.000000,0.000000,0.000000,10.500000,0.000000,0.000000,0.000000,11.500000,3.000000,0.000000,5.000000,0.000000
75%,2017-02-25 16:00:00,3.000000,29.000000,2.017023e+10,26.400000,85.000000,0.000000,0.000000,14.000000,0.000000,0.000000,0.000000,17.250000,5.000000,1.000000,9.000000,1.000000
max,2017-06-30 23:00:00,4.000000,180.000000,2.017063e+10,36.700000,100.000000,12.100000,12.100000,34.500000,1.000000,1.000000,1.000000,23.000000,6.000000,1.000000,12.000000,1.000000
std,NaN,0.966955,20.750063,5.944854e+06,4.558820,22.916549,0.502848,0.502848,4.961143,0.228687,0.169000,0.275572,6.922258,2.000017,0.451080,3.569872,0.454534


## 3. Data Cleaning & Pre Processing


### Handle Missing Values

In [6]:
# Check for missing values across all columns
print(df.isnull().sum())
print(f"Total missing values: {df.isnull().sum().sum()}")

DateTime            0
Junction            0
Vehicles            0
ID                  0
temperature_c       0
humidity_pct        0
precipitation_mm    0
rain_mm             0
wind_speed_kmh      0
is_holiday          0
is_event_day        0
is_special_day      0
hour                0
day_of_week         0
is_weekend          0
month               0
is_rush_hour        0
dtype: int64
Total missing values: 0


### Remove Duplicates

In [7]:
before = len(df)
df = df.drop_duplicates()
print(f"Rows before: {before}, after dedup: {len(df)}, removed: {before - len(df)}")

Rows before: 48120, after dedup: 48120, removed: 0


### Correct Data Types

In [8]:
# Convert categorical columns to category dtype for memory efficiency and correctness
df['Junction'] = df['Junction'].astype('category')

## 4. Aggrigate Traffic Data

In [9]:
dupe_check = df.duplicated(subset=['Junction', 'DateTime']).sum()
print("Duplicate (Junction, DateTime) pairs:", dupe_check)

# Hourly total across all junctions (useful for city-wide trend plots later)
hourly_total = df.groupby('DateTime', as_index=False)['Vehicles'].sum().rename(columns={'Vehicles': 'total_vehicles_all_junctions'})
hourly_total.head()

Duplicate (Junction, DateTime) pairs: 0


,DateTime,total_vehicles_all_junctions
0,2015-11-01 00:00:00,30
1,2015-11-01 01:00:00,26
2,2015-11-01 02:00:00,20
3,2015-11-01 03:00:00,14
4,2015-11-01 04:00:00,18


## 5. Preprocess the Data — Normalize / Standardize

In [10]:
# Standardize numeric columns; keep originals alongside scaled versions for later use
from sklearn.preprocessing import StandardScaler

scale_cols = ['Vehicles', 'temperature_c', 'humidity_pct', 'precipitation_mm', 'rain_mm', 'wind_speed_kmh']
scaler = StandardScaler()
scaled = scaler.fit_transform(df[scale_cols])
scaled_df = pd.DataFrame(scaled, columns=[f"{c}_scaled" for c in scale_cols], index=df.index)

df = pd.concat([df, scaled_df], axis=1)
df[[*scale_cols, *scaled_df.columns]].describe()

,Vehicles,temperature_c,humidity_pct,precipitation_mm,rain_mm,wind_speed_kmh,Vehicles_scaled,temperature_c_scaled,humidity_pct_scaled,precipitation_mm_scaled,rain_mm_scaled,wind_speed_kmh_scaled
count,48120.000000,48120.000000,48120.000000,48120.000000,48120.000000,48120.000000,4.812000e+04,4.812000e+04,4.812000e+04,4.812000e+04,4.812000e+04,4.812000e+04
mean,22.791334,23.520648,64.901517,0.096687,0.096687,11.272421,-1.039531e-16,3.024089e-16,-3.780111e-17,-3.898239e-17,-3.898239e-17,-4.725139e-18
std,20.750063,4.558820,22.916549,0.502848,0.502848,4.961143,1.000010e+00,1.000010e+00,1.000010e+00,1.000010e+00,1.000010e+00,1.000010e+00
min,1.000000,11.500000,9.000000,0.000000,0.000000,0.000000,-1.050193e+00,-2.636817e+00,-2.439377e+00,-1.922816e-01,-1.922816e-01,-2.272166e+00
25%,9.000000,20.300000,46.000000,0.000000,0.000000,7.900000,-6.646475e-01,-7.064727e-01,-8.248063e-01,-1.922816e-01,-1.922816e-01,-6.797740e-01
50%,15.000000,23.100000,68.000000,0.000000,0.000000,10.500000,-3.754888e-01,-9.227228e-02,1.352086e-01,-1.922816e-01,-1.922816e-01,-1.556958e-01
75%,29.000000,26.400000,85.000000,0.000000,0.000000,14.000000,2.992150e-01,6.316067e-01,8.770383e-01,-1.922816e-01,-1.922816e-01,5.497942e-01
max,180.000000,36.700000,100.000000,12.100000,12.100000,34.500000,7.576377e+00,2.890987e+00,1.531594e+00,2.387090e+01,2.387090e+01,4.681950e+00


## 6. Feature Engineering

### Time Based Features

In [11]:
df['hour'] = df['DateTime'].dt.hour
df['day_of_week'] = df['DateTime'].dt.dayofweek       # Monday=0 ... Sunday=6
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
df['month'] = df['DateTime'].dt.month
df['is_rush_hour'] = df['hour'].isin([8, 9, 10, 17, 18, 19, 20]).astype(int)
df[['DateTime', 'hour', 'day_of_week', 'is_weekend', 'month', 'is_rush_hour']].head()

,DateTime,hour,day_of_week,is_weekend,month,is_rush_hour
0,2015-11-01 00:00:00,0,6,1,11,0
1,2015-11-01 00:00:00,0,6,1,11,0
2,2015-11-01 00:00:00,0,6,1,11,0
3,2015-11-01 01:00:00,1,6,1,11,0
4,2015-11-01 01:00:00,1,6,1,11,0


### Lag Features

In [12]:
# Create lag features per junction so one junction's history doesn't leak into another's
df = df.sort_values(['Junction', 'DateTime']).reset_index(drop=True)

df['vehicles_lag_1h']  = df.groupby('Junction')['Vehicles'].shift(1)
df['vehicles_lag_24h'] = df.groupby('Junction')['Vehicles'].shift(24)
df['vehicles_lag_168h'] = df.groupby('Junction')['Vehicles'].shift(168)  # same hour, previous week

# Rolling average of the last 3 hours per junction (captures short-term trend)
df['vehicles_roll_mean_3h'] = (
    df.groupby('Junction')['Vehicles']
      .transform(lambda s: s.shift(1).rolling(window=3).mean())
)

print("Rows with NaN lag values (start of each junction's series):",
      df[['vehicles_lag_1h','vehicles_lag_24h','vehicles_lag_168h']].isna().any(axis=1).sum())
df[['Junction','DateTime','Vehicles','vehicles_lag_1h','vehicles_lag_24h','vehicles_lag_168h','vehicles_roll_mean_3h']].head(10)

Rows with NaN lag values (start of each junction's series): 672


,Junction,DateTime,Vehicles,vehicles_lag_1h,vehicles_lag_24h,vehicles_lag_168h,vehicles_roll_mean_3h
0,1,2015-11-01 00:00:00,15,NaN,NaN,NaN,NaN
1,1,2015-11-01 01:00:00,13,15.0,NaN,NaN,NaN
2,1,2015-11-01 02:00:00,10,13.0,NaN,NaN,NaN
3,1,2015-11-01 03:00:00,7,10.0,NaN,NaN,12.666667
4,1,2015-11-01 04:00:00,9,7.0,NaN,NaN,10.000000
5,1,2015-11-01 05:00:00,6,9.0,NaN,NaN,8.666667
6,1,2015-11-01 06:00:00,9,6.0,NaN,NaN,7.333333
7,1,2015-11-01 07:00:00,8,9.0,NaN,NaN,8.000000
8,1,2015-11-01 08:00:00,11,8.0,NaN,NaN,7.666667
9,1,2015-11-01 09:00:00,12,11.0,NaN,NaN,9.333333


### Total indicators for weekends & special events

In [13]:
print(df[['is_weekend','is_holiday','is_event_day','is_special_day']].sum())

is_weekend        13680
is_holiday         2664
is_event_day       1416
is_special_day     3984
dtype: int64


### Drop rows with NaN Lag Values

In [14]:
before = len(df)
df_model_ready = df.dropna(subset=['vehicles_lag_1h','vehicles_lag_24h','vehicles_lag_168h','vehicles_roll_mean_3h'])
print(f"Rows before: {before}, after dropping incomplete lag rows: {len(df_model_ready)}")

Rows before: 48120, after dropping incomplete lag rows: 47448


## 7. Feature Importance Evaluation

### Correlation with target (Vehicle)

In [15]:
numeric_cols = ['temperature_c','humidity_pct','precipitation_mm','rain_mm','wind_speed_kmh',
                'is_holiday','is_event_day','is_special_day','hour','day_of_week','is_weekend',
                'month','is_rush_hour','vehicles_lag_1h','vehicles_lag_24h','vehicles_lag_168h',
                'vehicles_roll_mean_3h']

corr = df_model_ready[numeric_cols + ['Vehicles']].corr()['Vehicles'].drop('Vehicles').sort_values(key=abs, ascending=False)
corr

,Vehicles
vehicles_lag_1h,0.970072
vehicles_roll_mean_3h,0.948963
vehicles_lag_168h,0.933112
vehicles_lag_24h,0.904835
hour,0.221029
temperature_c,0.201124
humidity_pct,-0.187434
is_weekend,-0.151111
day_of_week,-0.126301
is_rush_hour,0.066136


## 8. Save Preprocessed & Feature-Engineered Dataset

In [16]:
# Save the cleaned, aggregated, feature-engineered dataset — this is the Component 3 deliverable
output_cols = ['DateTime','Junction','Vehicles',
               'temperature_c','humidity_pct','precipitation_mm','rain_mm','wind_speed_kmh',
               'is_holiday','is_event_day','is_special_day',
               'hour','day_of_week','is_weekend','month','is_rush_hour',
               'vehicles_lag_1h','vehicles_lag_24h','vehicles_lag_168h','vehicles_roll_mean_3h']

df_model_ready[output_cols].to_csv('Traffic_Dataset_Preprocessed_FeatureEngineered_Yashas.csv', index=False)
print("Saved. Final shape:", df_model_ready[output_cols].shape)

Saved. Final shape: (47448, 20)
